In [1]:
!pip install feedparser
!pip install pandas
!pip install google-auth
!pip install google-api-python-client
!pip install requests
!pip install gspread
!pip install oauth2client
!pip install python-dotenv
!pip install jinja2

In [2]:
import re,os
import feedparser
import hashlib
from datetime import datetime, timezone
import pandas as pd
from google.auth import default
from googleapiclient.discovery import build
from google.oauth2 import service_account
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from dotenv import load_dotenv; load_dotenv()
import requests
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from email.utils import parsedate_to_datetime
from urllib.parse import urlparse, parse_qs, unquote

In [6]:
# 設定
SPREADSHEET_ID = "1WlamXyzIj6GZAkU_lc8C0mTvMzwoHZk-R_HodUC3Sws"
RANGE_IN_LIST_SHEET = "B2:B"  # worksheet("list").get(...) に渡す範囲

# HTMLタグ除去
def clean_html(text: str) -> str:
    text_without_tags = re.sub(r"<[^>]*>", " ", text or "")
    cleaned_text = re.sub(r"\s+", " ", text_without_tags)
    return cleaned_text.strip()

# GoogleリダイレクトURL除去
def clean_link(link: str) -> str:
    if not link:
        return ""
    try:
        p = urlparse(link)
        if "google.com" in p.netloc and p.path.startswith("/url"):
            q = parse_qs(p.query)
            for key in ("url", "q"):
                if key in q and q[key]:
                    return unquote(q[key][0])
    except Exception:
        pass
    return link

# スプレッドシートからコード一覧を取得（list!B2:B）
def get_code_from_spreadsheet() -> list[str]:
    scope = [
        "https://spreadsheets.google.com/feeds",
        "https://www.googleapis.com/auth/drive",
    ]
    credentials = ServiceAccountCredentials.from_json_keyfile_name(
        "abiding-ascent-476815-q6-56a05b29f113.json", scope
    )
    client = gspread.authorize(credentials)
    ws = client.open_by_key(SPREADSHEET_ID).worksheet("list")
    values = ws.get(RANGE_IN_LIST_SHEET)  # [["7203"], ["6758"], ...]
    return [row[0].strip() for row in values if row and row[0].strip()]

# 当日(JST)公表分のみを取得（publishedをそのまま利用）
def process_code_feed() -> pd.DataFrame:
    tokyo_tz = ZoneInfo("Asia/Tokyo")
    today_jst = datetime.now(tokyo_tz).date()

    codes = get_code_from_spreadsheet()
    rows = []
    
    for c in codes:
        for base in ("tdnet", "edinet"):
            url = f"https://webapi.yanoshin.jp/webapi/{base}/list/{c}.rss"
            try:
                feed = feedparser.parse(url)
            except Exception:
                continue

            for entry in feed.get("entries", []):
                published_raw = entry.get("published")
                if not published_raw:
                    continue
                # published は JST で提供される前提。tzinfo が無い場合は JST を付与するだけで変換はしない。
                try:
                    dt_pub = parsedate_to_datetime(published_raw)
                    if dt_pub.tzinfo is None:
                        dt_pub = dt_pub.replace(tzinfo=tokyo_tz)
                    pub_date_jst = dt_pub.date()
                except Exception:
                    # 解析できない場合はスキップ
                    continue
                if pub_date_jst == today_jst:
                    title = clean_html(str(entry.get("title", "")))
                    link = clean_link(str(entry.get("link", "")))
                    published = clean_html(published_raw)
                    rows.append({"title": title, "link": link, "published": published})

            

    if not rows:
        return pd.DataFrame(columns=["title", "link", "published"])

    df = pd.DataFrame(rows, columns=["title", "link", "published"])
    df = df.drop_duplicates(subset=["title", "link"]).reset_index(drop=True)
    return df

# 実行
df_code_list = process_code_feed()

In [7]:
df_code_list.head()

,title,link,published
0,ＤＩ:2026年３月期 第２四半期（中間期）決算短信〔日本基準〕（連結）,https://webapi.yanoshin.jp/rd.php?https://www.release.tdnet.info/inbs/140120251029581603.pdf,"Wed, 05 Nov 2025 17:30:00 +0900"
1,ＤＩ:2026年３月期 第２四半期決算説明会資料,https://webapi.yanoshin.jp/rd.php?https://www.release.tdnet.info/inbs/140120251029581605.pdf,"Wed, 05 Nov 2025 17:30:00 +0900"
2,ＤＩ:株式交付型インセンティブ・プランとしての自己株式処分に関するお知らせ,https://webapi.yanoshin.jp/rd.php?https://www.release.tdnet.info/inbs/140120251030583490.pdf,"Wed, 05 Nov 2025 17:30:00 +0900"
3,ＤＩ:株式交付型インセンティブ・プランへの追加拠出に関するお知らせ,https://webapi.yanoshin.jp/rd.php?https://www.release.tdnet.info/inbs/140120251030583494.pdf,"Wed, 05 Nov 2025 17:30:00 +0900"


In [8]:
df_display = df_code_list.copy()

# 改行や余計な空白を整理
df_display["title"] = df_display["title"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df_display["link"] = df_display["link"].astype(str).str.strip()

from IPython.display import display
pd.set_option("display.max_colwidth", None)

# Discord送信用（タイトルとリンクをまとめたテキスト）
entries = [
    f"{row['title']}\n{row['link']}"
    for _, row in df_display.iterrows()
]
text = "\n\n".join(entries)

# 環境変数がセットされていればそれを使い、なければ既存の変数を利用
WEBHOOK_URL = os.getenv("DISCORD_WEBHOOK_URL")

# Discordに送信
if text.strip():  # テキストが空でない場合のみ送信
    r = requests.post(WEBHOOK_URL, json={"content": text})